I got Cursor to write me a tracer. Let's try it out.

In [1]:
from tracer import Tracer
import sys

t = Tracer(max_spans=1000)
with t.trace():
    x = 0

    def foo(n):
        global x
        x = 0
        for i in range(n):
            bar(i)
        print("foo")

    def bar(k):
        global x
        x = k
        print(k, file=sys.stderr)

    def baz(a, b):
        return a + b

    def qux(x):
        return 2 * x

    foo(3)
    print(baz(qux(['x', 'y']), qux(['z'])))

def print_trace(trace):
	print("*"*50, f"Spans ({len(trace.spans)})", "*"*50)
	for span in trace.spans:
		print(span)

	print("*"*50, "Stdout", "*"*50)
	print(trace.stdout, end="")
	
	print("*"*50, "Stderr", "*"*50)
	print(trace.stderr, end="")

print_trace(t.traces()[0])

************************************************** Spans (39) **************************************************
type='line' line_number=6 line='x = 0' assignments={}
type='line' line_number=8 line='def foo(n):' assignments={}
type='line' line_number=15 line='def bar(k):' assignments={}
type='line' line_number=20 line='def baz(a, b):' assignments={}
type='line' line_number=23 line='def qux(x):' assignments={}
type='line' line_number=26 line='foo(3)' assignments={}
type='call' line_number=8 line='def foo(n):' arguments={'n': '3'}
type='line' line_number=10 line='x = 0' assignments={}
type='line' line_number=11 line='for i in range(n):' assignments={}
type='line' line_number=12 line='bar(i)' assignments={}
type='call' line_number=15 line='def bar(k):' arguments={'k': '0'}
type='line' line_number=17 line='x = k' assignments={}
type='line' line_number=18 line='print(k, file=sys.stderr)' assignments={}
type='return' line_number=18 line='print(k, file=sys.stderr)' return_value='None'
type='l

Let's try using the tracer on `openai/openai_humaneval`.

In [2]:
from datasets import load_dataset

ds = load_dataset("openai/openai_humaneval", split="test")

def format_src(prompt, canonical_solution, test, entry_point):
    return "\n".join([
        prompt,
        canonical_solution,
        test,
        f"check({entry_point})",
    ])

t.clear()
src = format_src(
	ds[0]["prompt"],
	ds[0]["canonical_solution"],
	ds[0]["test"],
	ds[0]["entry_point"],
)
t.trace_str(src)

print_trace(t.traces()[0])

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

************************************************** Spans (568) **************************************************
type='line' line_number=1 line='from typing import List' assignments={}
type='line' line_number=4 line='def has_close_elements(numbers: List[float], threshold: float) -> bool:' assignments={}
type='line' line_number=25 line="'author': 'jt'," assignments={}
type='line' line_number=26 line="'dataset': 'test'" assignments={}
type='line' line_number=24 line='METADATA = {' assignments={}
type='line' line_number=30 line='def check(candidate):' assignments={}
type='line' line_number=40 line='check(has_close_elements)' assignments={}
type='call' line_number=30 line='def check(candidate):' arguments={}
type='line' line_number=31 line='assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True' assignments={}
type='call' line_number=4 line='def has_close_elements(numbers: List[float], threshold: float) -> bool:' arguments={'numbers': '[1.0, 2.0, 3.9, 4.0, 5.0, 2.2]', 'threshold': 

Perfect. Let's do the rest now.

In [3]:
from tqdm import tqdm

t.clear()
for prompt, canonical_solution, test, entry_point in tqdm(zip(
    ds["prompt"],
    ds["canonical_solution"],
    ds["test"],
    ds["entry_point"],
), total=len(ds)):
    src = format_src(prompt, canonical_solution, test, entry_point)
    t.trace_str(src)
traces = t.traces()

print_trace(traces[0])

100%|██████████| 164/164 [00:00<00:00, 263.99it/s]

************************************************** Spans (568) **************************************************
type='line' line_number=1 line='from typing import List' assignments={}
type='line' line_number=4 line='def has_close_elements(numbers: List[float], threshold: float) -> bool:' assignments={}
type='line' line_number=25 line="'author': 'jt'," assignments={}
type='line' line_number=26 line="'dataset': 'test'" assignments={}
type='line' line_number=24 line='METADATA = {' assignments={}
type='line' line_number=30 line='def check(candidate):' assignments={}
type='line' line_number=40 line='check(has_close_elements)' assignments={}
type='call' line_number=30 line='def check(candidate):' arguments={}
type='line' line_number=31 line='assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True' assignments={}
type='call' line_number=4 line='def has_close_elements(numbers: List[float], threshold: float) -> bool:' arguments={'numbers': '[1.0, 2.0, 3.9, 4.0, 5.0, 2.2]', 'threshold': 

Great. Now, let's fine tune the model.

In [4]:
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, AddedToken
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-Coder-3B"
SPANS_START = "<spans>"
SPANS_END = "</spans>"

def format_span(span):
    return str(span)

def format_trace(trace):
    return "\n".join([
        SPANS_START,
        *[format_span(span) for span in trace.spans],
        SPANS_END,
    ])

examples = []
for prompt, canonical_solution, test, entry_point, trace in zip(
    ds["prompt"],
    ds["canonical_solution"],
    ds["test"],
    ds["entry_point"],
    traces,
):
    examples.append({
        "prompt": format_src(prompt, canonical_solution, test, entry_point) + "\n",
        "completion": format_trace(trace),
    })
train_ds = Dataset.from_list(examples)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.add_special_tokens({
    "additional_special_tokens": [
        AddedToken(SPANS_START, special=True, normalized=False),
        AddedToken(SPANS_END, special=True, normalized=False),
    ],
})
model.resize_token_embeddings(len(tokenizer))

args = SFTConfig(
    output_dir="trainer_output",
    completion_only_loss=True,
    max_length=8192,
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    processing_class=tokenizer,
)
trainer.train()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

Building labels for train dataset:   0%|          | 0/164 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/164 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/164 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/ethankim8683/Machine Learning/cwm/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: Invalid buffer size: 32.00 GiB